In [31]:
import os

In [32]:
from dotenv import load_dotenv

In [33]:
from pydantic import BaseModel, Field

In [34]:
from pydantic import BaseModel, Field

In [35]:
from langchain_groq import ChatGroq

In [36]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

In [37]:
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

In [38]:
load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [39]:
class ReviewData(BaseModel):
    complaint: str = Field(description="Main customer complaint")
    product_or_feature: str = Field(description="Product or feature mentioned")
    sentiment: str = Field(description="Customer sentiment")


parser = PydanticOutputParser(pydantic_object=ReviewData)

In [40]:
prompt1 = PromptTemplate(
    template="""
Analyze the customer review and extract the complaint,
product or feature, and sentiment.

Customer Review:
{review_text}

{format_instructions}
""",
    input_variables=["review_text"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

chain1 = prompt1 | llm | parser

In [41]:
prompt2 = ChatPromptTemplate.from_messages([
    ("system", "You are a support ticket writer."),
    ("human", """
Create a short support ticket summary for the internal team.

Complaint: {complaint}
Product or Feature: {product_or_feature}
Sentiment: {sentiment}
""")
])

chain2 = prompt2 | llm | StrOutputParser()

In [42]:
load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [ ]:
review_text = input("Enter customer review: ")

chain1 = prompt1 | llm | parser
chain2 = prompt2 | llm | StrOutputParser()

result1 = chain1.invoke({
    "review_text": review_text
})

print("\nStep 1 - Structured Complaint Data")
print(result1.model_dump())

result2 = chain2.invoke({
    "complaint": result1.complaint,
    "product_or_feature": result1.product_or_feature,
    "sentiment": result1.sentiment
})

print("\nStep 2 - Support Ticket Summary")
print(result2)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}